## AI Search

In [4]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
import os
import json
import requests

In [5]:
load_dotenv(override=True) # take environment variables from .env.

def check_empty(variable_name, value):  
    if not value:  
        print(f"{variable_name} is empty.")  
  
AZURE_SEARCH_SERVICE_NAME = os.getenv("AZURE_SEARCH_SERVICE_NAME")

AZURE_SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX") 
AZURE_SEARCH_INDEX_LAYOUT = os.getenv("AZURE_SEARCH_INDEX_LAYOUT")
AZURE_SEARCH_INDEX_OCR = os.getenv("AZURE_SEARCH_INDEX_OCR")

AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
BLOB_CONNECTION_STRING = os.getenv("BLOB_CONNECTION_STRING")
BLOB_CONTAINER_NAME = os.getenv("BLOB_CONTAINER_NAME") 

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_EMBEDDING_MODEL_NAME = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL_NAME")
AZURE_OPENAI_EMBEDDING_DIMENSIONS = os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS")
AZURE_AI_SERVICES_ENDPOINT = os.getenv("AZURE_AI_SERVICES_ENDPOINT")

check_empty("AZURE_SEARCH_SERVICE_NAME", AZURE_SEARCH_SERVICE_NAME)
check_empty("AZURE_SEARCH_INDEX", AZURE_SEARCH_INDEX)
check_empty("AZURE_SEARCH_API_KEY", AZURE_SEARCH_API_KEY)
check_empty("BLOB_CONNECTION_STRING", BLOB_CONNECTION_STRING)
check_empty("BLOB_CONTAINER_NAME", BLOB_CONTAINER_NAME)
check_empty("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT)
check_empty("AZURE_OPENAI_EMBEDDING_MODEL_NAME", AZURE_OPENAI_EMBEDDING_MODEL_NAME)
check_empty("AZURE_OPENAI_EMBEDDING_DIMENSIONS", AZURE_OPENAI_EMBEDDING_DIMENSIONS)
check_empty("AZURE_SEARCH_INDEX", AZURE_SEARCH_INDEX)
check_empty("AZURE_AI_SERVICES_ENDPOINT", AZURE_AI_SERVICES_ENDPOINT)





## Create Blob Storage

In [6]:
def create_datasource(service_name, index_name, search_api_key, storage_connectionstring, storage_container):
    print("AZURE_SEARCH_INDEX = ", AZURE_SEARCH_INDEX)
    print("index_name = ", index_name)
    endpoint = "https://{}.search.windows.net/".format(service_name)
    url = '{0}/datasources/{1}-datasource?api-version=2024-11-01-preview'.format(endpoint, index_name)

    payload = json.dumps({
        "name": index_name + "-datasource",
        "description": None,
        "type": "azureblob",
        "subtype": None,
        "credentials": {
            "connectionString": storage_connectionstring
        },
        "container": {
            "name": storage_container,
            "query": None
        },
        "dataChangeDetectionPolicy": None,
        "dataDeletionDetectionPolicy": {
            "@odata.type": "#Microsoft.Azure.Search.NativeBlobSoftDeleteDeletionDetectionPolicy"
        },
        "encryptionKey": None,
        "identity": None
        })

    headers = {
    'api-key': search_api_key,
    'Content-Type': 'application/json'
            }


    response = requests.request("PUT", url, headers=headers, data=payload)

    if response.status_code == 201 or response.status_code == 204:
        return response, True
    else:
        print("Error creating data source: ", response.text)
        return response, False

## Create a Search Index

In [54]:
def create_index_semantic(service_name, index,azure_search_api_key, openai_api_base, text_embbeding_model, text_embbeding_model_dimensions):

    endpoint = "https://{}.search.windows.net/".format(service_name)
    url = '{0}/indexes/{1}/?api-version=2024-11-01-preview'.format(endpoint, index)

    resourceUri = openai_api_base
    deploymentId =  text_embbeding_model

    print(url)

    payload = json.dumps({
        "name": index,
          "fields": [
            {
            "name": "chunk_id",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": True,
            "facetable": False,
            "key": True,
            "analyzer": "keyword"
            },
            {
            "name": "parent_id",
            "type": "Edm.String",
            "searchable": False,
            "filterable": True,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "chunk",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "title",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "header_1",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "header_2",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "header_3",
            "type": "Edm.String",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False
            },
            {
            "name": "loads",
            "type": "Collection(Edm.String)",
            "searchable": True,
            "filterable": True,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False,
            },
            {
            "name": "text_vector",
            "type": "Collection(Edm.Single)",
            "searchable": True,
            "filterable": False,
            "retrievable": True,
            "stored": True,
            "sortable": False,
            "facetable": False,
            "key": False,
            "dimensions": text_embbeding_model_dimensions,

            "vectorSearchProfile":"{index}-azureOpenAi-text-profile".format(index=index)
            }
        ],
        "scoringProfiles": [],
        "suggesters": [],
        "analyzers": [],
        "normalizers": [],
        "tokenizers": [],
        "tokenFilters": [],
        "charFilters": [],
        "similarity": {
            "@odata.type": "#Microsoft.Azure.Search.BM25Similarity"
        },
        "semantic": {
            "configurations": [
            {
                "name": "{index}-semantic-configuration".format(index=index),
                "prioritizedFields": {
                "titleField": {
                    "fieldName": "title"
                },
                "prioritizedContentFields": [
                    {
                    "fieldName": "chunk"
                    }
                ],
                "prioritizedKeywordsFields": []
                }
            }
            ]
        },
        "vectorSearch": {
            "algorithms": [
            {
                "name": "{index}-algorithm".format(index=index),
                "kind": "hnsw",
                "hnswParameters": {
                "metric": "cosine",
                "m": 4,
                "efConstruction": 400,
                "efSearch": 500
                }
            }
            ],
            "profiles": [
            {
                "name": "{index}-azureOpenAi-text-profile".format(index=index),
                "algorithm": "{index}-algorithm".format(index=index),
                "vectorizer": "{index}-azureOpenAi-text-vectorizer".format(index=index),
            }
            ],
            "vectorizers": [
            {
                "name": "{index}-azureOpenAi-text-vectorizer".format(index=index),
                "kind": "azureOpenAI",
                "azureOpenAIParameters": {
                "resourceUri": openai_api_base,
                "deploymentId": deploymentId,
                "modelName": deploymentId
                }
            }
            ],
            "compressions": []
        }
        })
    headers = {
    'api-key': azure_search_api_key,
    'Content-Type': 'application/json'
    }

    response = requests.request("PUT", url, headers=headers, data=payload)

    if response.status_code == 201 or response.status_code == 204:
        return response, True
    else:
        print('************************')
        print(response.status_code)
        print(response.text)
        return response, False

## Create a Skillset

- AI Skill can be used

```
        "cognitiveServices": {
          "@odata.type": "#Microsoft.Azure.Search.AIServicesByIdentity",
          "subdomainUrl": azure_ai_services_endpoint
        },
```

In [55]:
def create_skillset(service_name, search_api_key, index, openai_api_base, text_embbeding_model, azure_ai_services_endpoint, text_embbeding_model_dimensions):

    endpoint = "https://{}.search.windows.net/".format(service_name)
    resourceUri = openai_api_base
    deploymentId =  text_embbeding_model
    modelName =  text_embbeding_model
    dimensions = text_embbeding_model_dimensions
    embeddingFunctionAppUriAndKey = os.getenv("AZURE_FUNCTION_APP_URL")
    azure_ai_services_key = os.getenv("AZURE_AI_SERVICES_KEY")


    url = '{0}/skillsets/{1}-skillset?api-version=2024-11-01-preview'.format(endpoint, index)
    print(url)
    payload = json.dumps(
       {
        "name": index + "-skillset",
        "description": "Skillset to chunk documents and generate embeddings",
        "skills": [
          {
            "@odata.type": "#Microsoft.Skills.Util.DocumentIntelligenceLayoutSkill",
            "name": "#1",
            "context": "/document",
            "outputMode": "oneToMany",
            "markdownHeaderDepth": "h3",
            "inputs": [
              {
                "name": "file_data",
                "source": "/document/file_data",
                "inputs": []
              }
            ],
            "outputs": [
              {
                "name": "markdown_document",
                "targetName": "markdownDocument"
              }
            ]
          },
          {
                        "@odata.type": "#Microsoft.Skills.Custom.WebApiSkill",
                        "uri": embeddingFunctionAppUriAndKey,
                        "httpMethod": "POST",
                        "timeout": "PT230S",
                        "batchSize": 1,
                        "degreeOfParallelism": 1,
                        "name": "LoadFinder",
                        "description": "",
                        "context": "/document",
                        "inputs": [
                              {
                                "name": "text",
                                "source": "/document/markdownDocument",
                                "inputs": []
                              }
                            ],
                            "outputs": [
                              {
                                "name": "loads",
                                "targetName": "loads"
                              }
                            ]
                      },
          {
            "@odata.type": "#Microsoft.Skills.Text.SplitSkill",
            "name": "#2",
            "description": "Split skill to chunk documents",
            "context": "/document/markdownDocument/*",
            "defaultLanguageCode": "en",
            "textSplitMode": "pages",
            "maximumPageLength": 2000,
            "pageOverlapLength": 500,
            "maximumPagesToTake": 0,
            "unit": "characters",
            "inputs": [
              {
                "name": "text",
                "source": "/document/markdownDocument/*/content",
                "inputs": []
              }
            ],
            "outputs": [
              {
                "name": "textItems",
                "targetName": "pages"
              }
            ]
          },
          {
            "@odata.type": "#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill",
            "name": "#3",
            "context": "/document/markdownDocument/*/pages/*",
            "resourceUri": openai_api_base,
            "deploymentId": text_embbeding_model,
            "dimensions": text_embbeding_model_dimensions,
            "modelName": text_embbeding_model,
            "inputs": [
              {
                "name": "text",
                "source": "/document/markdownDocument/*/pages/*",
                "inputs": []
              }
            ],
            "outputs": [
              {
                "name": "embedding",
                "targetName": "text_vector"
              }
            ]
          }
        ],
        "cognitiveServices": {
          "@odata.type": "#Microsoft.Azure.Search.AIServicesByKey",
          "subdomainUrl": azure_ai_services_endpoint,
          "key": azure_ai_services_key
        },
        "indexProjections": {
          "selectors": [
            {
              "targetIndexName": index,
              "parentKeyFieldName": "parent_id",
              "sourceContext": "/document/markdownDocument/*/pages/*",
              "mappings": [
                {
                  "name": "text_vector",
                  "source": "/document/markdownDocument/*/pages/*/text_vector",
                  "inputs": []
                },
                {
                  "name": "chunk",
                  "source": "/document/markdownDocument/*/pages/*",
                  "inputs": []
                },
                {
                  "name": "title",
                  "source": "/document/title",
                  "inputs": []
                },
                  {
                  "name": "loads",
                  "source": "/document/loads",
                  "inputs": []
                },
                {
                  "name": "header_1",
                  "source": "/document/markdownDocument/*/sections/h1",
                  "inputs": []
                },
                {
                  "name": "header_2",
                  "source": "/document/markdownDocument/*/sections/h2",
                  "inputs": []
                },
                {
                  "name": "header_3",
                  "source": "/document/markdownDocument/*/sections/h3",
                  "inputs": []
                }
              ]
            }
          ],
          "parameters": {
            "projectionMode": "skipIndexingParentDocuments"
          }
        }
      }
    )
    
    headers = {
        'Content-Type': 'application/json',
        'api-key': '{0}'.format(search_api_key)
    }


    
    response = requests.request("PUT", url, headers=headers, data=payload)


    if response.status_code == 201 or response.status_code == 204:
        return response, True
    else:
        print(response.text)
        return response, False

## Create Indexer

In [56]:
def create_indexer(service_name, index, search_key):
        endpoint = "https://{}.search.windows.net/".format(service_name)
        url = '{0}/indexers/{1}-indexer/?api-version=2024-11-01-preview'.format(endpoint, index)
        print(url)

        payload = json.dumps({
        "name": "{0}-indexer".format(index),
        "description": None,
        "dataSourceName": "{0}-datasource".format(index),
        "skillsetName": "{0}-skillset".format(index),
        "targetIndexName": "{0}".format(index),
        "disabled": None,
        "schedule": None,
        "parameters": {
            "batchSize": None,
            "maxFailedItems": None,
            "maxFailedItemsPerBatch": None,
            "base64EncodeKeys": None,
            "configuration": {
            "dataToExtract": "contentAndMetadata",
            "parsingMode": "default",
            "allowSkillsetToReadFileData": True
            }
        },
        "fieldMappings": [
            {
            "sourceFieldName": "metadata_storage_name",
            "targetFieldName": "title",
            "mappingFunction": None
            }
        ],
        "outputFieldMappings": [
        ],
        "cache": None,
        "encryptionKey": None
        })
        headers = {
        'Content-Type': 'application/json',
        'api-key': '{0}'.format(search_key)
        }


        response = requests.request("PUT", url, headers=headers, data=payload)


        if response.status_code == 201 or response.status_code == 204:
            print('good')
            return response, True
        else:
            print(response.status_code)
            print('************************')
            print(response.status_code)
            print(response.text)
            return response, False

In [57]:
create_datasource(AZURE_SEARCH_SERVICE_NAME, AZURE_SEARCH_INDEX, AZURE_SEARCH_API_KEY, BLOB_CONNECTION_STRING, BLOB_CONTAINER_NAME)

AZURE_SEARCH_INDEX =  vector-manual13
index_name =  vector-manual13


(<Response [201]>, True)

In [58]:

create_index_semantic(AZURE_SEARCH_SERVICE_NAME, AZURE_SEARCH_INDEX,AZURE_SEARCH_API_KEY, AZURE_OPENAI_ENDPOINT,  AZURE_OPENAI_EMBEDDING_MODEL_NAME, AZURE_OPENAI_EMBEDDING_DIMENSIONS)

https://mmchrsearch02.search.windows.net//indexes/vector-manual13/?api-version=2024-11-01-preview


(<Response [201]>, True)

In [59]:
#service_name, search_api_key, index, openai_api_base, text_embbeding_model, azure_ai_services_endpoint
create_skillset(AZURE_SEARCH_SERVICE_NAME, AZURE_SEARCH_API_KEY, AZURE_SEARCH_INDEX,  AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_EMBEDDING_MODEL_NAME, AZURE_AI_SERVICES_ENDPOINT, AZURE_OPENAI_EMBEDDING_DIMENSIONS)

https://mmchrsearch02.search.windows.net//skillsets/vector-manual13-skillset?api-version=2024-11-01-preview


(<Response [201]>, True)

In [60]:
#create_indexer(service_name, index, search_key)
create_indexer(AZURE_SEARCH_SERVICE_NAME, AZURE_SEARCH_INDEX, AZURE_SEARCH_API_KEY)

https://mmchrsearch02.search.windows.net//indexers/vector-manual13-indexer/?api-version=2024-11-01-preview
good


(<Response [201]>, True)

## Structured Output
https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/structured-outputs?tabs=python%2Cdotnet-entra-id&pivots=programming-language-python

In [15]:
from pydantic import BaseModel
from openai import AzureOpenAI
import os

client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version="2024-10-21"
)

class Results(BaseModel):
    Loads: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-4o", # replace with the model deployment name of your gpt-4o 2024-08-06 deployment
    messages=[
        {"role": "system", "content": "Objective: Extract unique load information from bills of lading and compile them into a distinct, structured list. A 'load' refers to the specific cargo or shipment details (e.g., item descriptions, quantities, or identifiers). The output should exclude duplicates and irrelevant information."},
        {"role": "user", "content": "I need to ship several loads  I need to ship load L12345, and L4567."},
    ],
    response_format=Results,
)

event = completion.choices[0].message.parsed

print(event.Loads)


['L12345', 'L4567']
